# Hyperparameter Tuning: DeepDetect
The goal of this experiment is to find the best hyperparameter configuration that balances the frequency and spatial branches. The parameters tuned are: backbone learning rate (separate from the rest of the network), main learning rate, frequency auxiliary loss weight, diversity regulariser weight, and gate initial bias. We use Optuna with TPE (Tree-structured Parzen Estimator) sampler, a sequential model-based optimisation algorithm that learns from previous trials to focus search on promising regions. MedianPruner stops unpromising trials early based on intermediate validation accuracy. Tuning is performed on convnext_base with gating fusion and the resulting configuration is applied uniformly to all backbones, since using consistent hyperparameters across backbones ensures that performance differences are attributable to architecture.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell


In [ ]:
# Install optuna 
import subprocess
subprocess.run(["pip", "install", "optuna", "-q"], check=True)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"Optuna version: {optuna.__version__}")


Optuna version: 4.8.0


## Config and Data

In [ ]:
from config import Config
from data.deepdetect import get_deepdetect_loaders

def make_base_cfg():
    cfg = Config()
    cfg.data.deepdetect_root   = "../data/raw/deep_detect/data"
    cfg.data.dataset           = "deepdetect"
    cfg.data.image_size        = 224
    cfg.data.batch_size        = 88
    cfg.data.num_workers       = 4
    cfg.backbone.name          = "convnext_base"
    cfg.backbone.pretrained    = True
    cfg.backbone.img_size      = 224
    cfg.fusion.mode            = "gating"
    cfg.train.epochs           = 10        # short runs for tuning
    # Augmentations — all on for DeepDetect
    cfg.data.jpeg_aug          = True
    cfg.data.blur_aug          = True
    cfg.data.noise_aug         = True
    cfg.data.recompression_aug = True
    cfg.data.mixed_aug         = True
    cfg.data.mixed_aug_prob    = 0.3
    return cfg

# Load data 
cfg_data = make_base_cfg()
train_loader, val_loader, test_loader = get_deepdetect_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776


## Objective Function

In [4]:
from experiments.train import train
import gc

def objective(trial):
    cfg = make_base_cfg()

    # Hyperparameter search space
    cfg.train.backbone_lr       = trial.suggest_float("backbone_lr",      1e-6, 5e-5, log=True)
    cfg.train.lr                = trial.suggest_float("lr",                5e-5, 5e-4, log=True)
    cfg.loss.freq_aux_weight    = trial.suggest_float("freq_aux_weight",   0.3,  1.0)
    cfg.fusion.diversity_weight = trial.suggest_float("diversity_weight",  0.05, 0.3)
    cfg.fusion.gate_init_bias   = trial.suggest_float("gate_init_bias",    0.0,  0.3)

    cfg.experiment_name = f"tune_trial_{trial.number}"
    cfg.notes           = f"optuna trial {trial.number}"

    try:
        best_val_acc = train(cfg, train_loader, val_loader)

        # Prune if val accuracy is too low after 10 epochs
        trial.report(best_val_acc, step=cfg.train.epochs)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        return best_val_acc

    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return 0.0
    finally:
        # Free GPU memory between trials
        torch.cuda.empty_cache()
        gc.collect()


## Run Tuning

In [5]:
def print_callback(study, trial):
    print(f"Trial {trial.number} complete | val_acc={trial.value:.1%} | "
          f"best so far={study.best_value:.1%}")
    print(f"  params: {trial.params}")

study = optuna.create_study(
    study_name="asfr_deepdetect_tuning",
    storage="sqlite:///optuna_deepdetect.db",
    load_if_exists=True,               # resumes if db already exists
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True,           # widget causes issues in Jupyter
    callbacks=[print_callback],        # prints one line per completed trial
)

print(f"\nBest trial: {study.best_trial.number}")
print(f"Best val accuracy: {study.best_value:.1%}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")


  0%|          | 0/20 [00:00<?, ?it/s]

Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell



Experiment: tune_trial_1
Backbone: convnext_base | Fusion: gating | Frozen: False
Epochs: 10 | LR: 0.0004463590152176814 | Batch: 88



Epoch   1/10 | train_loss=0.9232 | val_acc=93.3% | val_auc=0.996 | val_f1=0.922
  gate — entropy=2.257 nats | mean=0.440 | var=0.0147
  grad norms — freq=0.0161 | spatial=0.9998
  -> Saved best val_acc=93.3%


Epoch   2/10 | train_loss=0.7138 | val_acc=96.0% | val_auc=0.998 | val_f1=0.955
  gate — entropy=2.126 nats | mean=0.426 | var=0.0105
  -> Saved best val_acc=96.0%


...

Epoch   6/10 | train_loss=0.7058 | val_acc=96.9% | val_auc=1.000 | val_f1=0.965
  gate — entropy=2.030 nats | mean=0.490 | var=0.0084
  grad norms — freq=0.4233 | spatial=0.9059


Epoch   7/10 | train_loss=0.6955 | val_acc=98.4% | val_auc=0.999 | val_f1=0.983
  gate — entropy=2.280 nats | mean=0.628 | var=0.0152


Epoch   8/10 | train_loss=0.6872 | val_acc=98.7% | val_auc=0.999 | val_f1=0.986
  gate — entropy=2.307 nats | mean=0.670 | var=0.0165
  -> Saved best val_acc=98.7%


Epoch   9/10 | train_loss=0.6823 | val_acc=98.6% | val_auc=0.999 | val_f1=0.984
  gate — entropy=2.311 nats | mean=0.649 | var=0.0166


Epoch  10/10 | train_loss=0.6796 | val_acc=97.9% | val_auc=0.999 | val_f1=0.977
  gate — entropy=2.377 nats | mean=0.645 | var=0.0193

Training complete. Best val accuracy: 98.7%
Results will be logged to CSV after full_evaluation().
Trial 20 failed: 
Trial 20 complete | val_acc=0.0% | best so far=99.8%
  params: {'backbone_lr': 6.676232043066747e-06, 'lr': 0.00022947755762516922, 'freq_aux_weight': 0.9784888964341025, 'diversity_weight': 0.11449093438061776, 'gate_init_bias': 0.22580951071275235}

Best trial: 13
Best val accuracy: 99.8%
Best params:
  backbone_lr: 0.000049
  lr: 0.000279
  freq_aux_weight: 0.500098
  diversity_weight: 0.058435
  gate_init_bias: 0.215438


## Results

In [6]:
import pandas as pd

df = study.trials_dataframe()
cols = ["number", "value"] + [c for c in df.columns if c.startswith("params_")]
print(df[cols].sort_values("value", ascending=False).head(10).to_string(index=False))


 number    value  params_backbone_lr  params_diversity_weight  params_freq_aux_weight  params_gate_init_bias  params_lr
     13 0.998378            0.000049                 0.058435                0.500098               0.215438   0.000279
     14 0.998083            0.000044                 0.051956                0.509508               0.245477   0.000271
     19 0.998083            0.000028                 0.080486                0.552308               0.246142   0.000346
     12 0.997788            0.000039                 0.071396                0.467587               0.189463   0.000248
     17 0.997788            0.000050                 0.132645                0.570121               0.298540   0.000320
     11 0.997567            0.000046                 0.054490                0.475090               0.192027   0.000251
     15 0.996092            0.000027                 0.102595                0.577671               0.261265   0.000288
     18 0.996092            0.000016    

## Save Best Config

In [8]:
best = study.best_params

print("\nBest hyperparameters: ")
print(f'cfg.train.backbone_lr       = {best["backbone_lr"]:.2e}')
print(f'cfg.train.lr                = {best["lr"]:.2e}')
print(f'cfg.loss.freq_aux_weight    = {best["freq_aux_weight"]:.3f}')
print(f'cfg.fusion.diversity_weight = {best["diversity_weight"]:.3f}')
print(f'cfg.fusion.gate_init_bias   = {best["gate_init_bias"]:.3f}')



Best hyperparameters: 
cfg.train.backbone_lr       = 4.85e-05
cfg.train.lr                = 2.79e-04
cfg.loss.freq_aux_weight    = 0.500
cfg.fusion.diversity_weight = 0.058
cfg.fusion.gate_init_bias   = 0.215
